# 🌊 SWACHH-NAYAN: Drain Waste AI Model Training & RV1103 NPU Export
### Edge-Vision Storm-Drain Trash & Micro-Plastic Diverter
This turnkey Google Colab notebook trains **YOLOv5n** on a storm drain waste dataset and exports it with **Rockchip RV1103 NPU subgraph optimization** for the Luckfox Pico.

In [ ]:
# Step 1: Verify GPU Allocation
!nvidia-smi

In [ ]:
# Step 2: Clone Rockchip-optimized YOLOv5 repository (airockchip)
import os
%cd /content
if not os.path.exists('/content/yolov5'):
    !git clone https://github.com/airockchip/yolov5.git /content/yolov5
%cd /content/yolov5
!pip install -r requirements.txt
!pip install onnx onnxscript gdown roboflow

In [ ]:
# Step 3: CRITICAL PATCH - Fix PyTorch 2.6+ / 2.11 weights_only=True default in Colab
# PyTorch 2.6+ blocks unpickling models.yolo.Model without weights_only=False.
patch_code = """import torch
# Global PyTorch 2.6+ weights_only compatibility patch
_orig_torch_load = torch.load
torch.load = lambda *args, **kwargs: _orig_torch_load(*args, **{**kwargs, 'weights_only': False})
"""

for py_file in ['train.py', 'export.py', 'val.py', 'detect.py', 'models/yolo.py']:
    if os.path.exists(py_file):
        with open(py_file, 'r') as f:
            content = f.read()
        if '_orig_torch_load' not in content:
            with open(py_file, 'w') as f:
                f.write(patch_code + '\n' + content)

print("✅ Successfully applied PyTorch 2.6+ compatibility patch to YOLOv5 scripts!")

In [ ]:
# Step 4: Define Drain Waste Dataset YAML Configuration
dataset_yaml = """
path: /content/drain_waste_dataset
train: images/train
val: images/val
test: images/val

nc: 6
names: ['plastic_bottle', 'plastic_bag', 'aluminum_can', 'styrofoam_cup', 'organic_debris', 'microplastic_fragment']
"""
os.makedirs('/content/yolov5/data', exist_ok=True)
with open('/content/yolov5/data/drain_waste.yaml', 'w') as f:
    f.write(dataset_yaml.strip())
print("✅ Created /content/yolov5/data/drain_waste.yaml")

In [ ]:
# Step 5: Acquire / Prepare Drain Waste Dataset
import os, random
import numpy as np
from PIL import Image, ImageDraw

# Create directory structure
for split in ['train', 'val']:
    os.makedirs(f'/content/drain_waste_dataset/images/{split}', exist_ok=True)
    os.makedirs(f'/content/drain_waste_dataset/labels/{split}', exist_ok=True)

# Check if user wants to download external dataset (e.g. from Google Drive or Roboflow)
# If empty, we automatically generate synthetic drain waste training samples so training completes immediately!
train_imgs = os.listdir('/content/drain_waste_dataset/images/train')
if len(train_imgs) == 0:
    print("Generating initial training dataset samples with water flume backgrounds & annotated drain waste...")
    class_colors = [(0, 120, 255), (255, 255, 255), (192, 192, 192), (255, 230, 200), (34, 139, 34), (255, 0, 128)]
    
    # Generate training & validation samples including HARD BACKGROUND NEGATIVES (empty water ripples/foam/reflections)\n    # Background images with empty label files explicitly teach YOLO that water motion has NO OBJECTS!\n    for split, waste_count, bg_count in [('train', 110, 40), ('val', 30, 10)]:\n        # 1. Images with Drain Waste Material\n        for i in range(waste_count):\n            img = Image.new('RGB', (640, 640), color=(random.randint(20, 50), random.randint(70, 110), random.randint(60, 90)))\n            draw = ImageDraw.Draw(img)\n            for _ in range(random.randint(5, 15)):\n                rx = random.randint(0, 600)\n                ry = random.randint(0, 630)\n                rw = random.randint(30, 100)\n                draw.arc([rx, ry, rx + rw, ry + 15], 0, 180, fill=(random.randint(50, 90), random.randint(110, 150), random.randint(100, 140)), width=1)\n            labels = []\n            num_objs = random.randint(1, 3)\n            for _ in range(num_objs):\n                cls_id = random.randint(0, 5)\n                w = random.randint(45, 150)\n                h = random.randint(45, 150)\n                x1 = random.randint(50, 640 - w - 50)\n                y1 = random.randint(50, 640 - h - 50)\n                x2 = x1 + w\n                y2 = y1 + h\n                draw.rectangle([x1, y1, x2, y2], fill=class_colors[cls_id], outline=(255, 255, 255), width=2)\n                cx = (x1 + w / 2) / 640.0\n                cy = (y1 + h / 2) / 640.0\n                nw = w / 640.0\n                nh = h / 640.0\n                labels.append(f\"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\")\n            img.save(f'/content/drain_waste_dataset/images/{split}/sample_waste_{i:04d}.jpg')\n            with open(f'/content/drain_waste_dataset/labels/{split}/sample_waste_{i:04d}.txt', 'w') as f:\n                f.write('\\n'.join(labels))\n        # 2. Hard Negative Backgrounds (pure flowing water, ripples, waves, glare, zero waste)\n        for i in range(bg_count):\n            img = Image.new('RGB', (640, 640), color=(random.randint(20, 50), random.randint(70, 110), random.randint(60, 90)))\n            draw = ImageDraw.Draw(img)\n            for _ in range(random.randint(20, 45)):\n                rx = random.randint(0, 600)\n                ry = random.randint(0, 630)\n                rw = random.randint(40, 180)\n                draw.arc([rx, ry, rx + rw, ry + random.randint(10, 25)], 0, 180, fill=(random.randint(60, 110), random.randint(120, 170), random.randint(110, 160)), width=random.randint(1, 3))\n            gx = random.randint(50, 500)\n            gy = random.randint(50, 500)\n            draw.ellipse([gx, gy, gx + 80, gy + 30], fill=(random.randint(80, 130), random.randint(130, 180), random.randint(120, 170)))\n            img.save(f'/content/drain_waste_dataset/images/{split}/sample_bg_{i:04d}.jpg')\n            with open(f'/content/drain_waste_dataset/labels/{split}/sample_bg_{i:04d}.txt', 'w') as f:\n                f.write('')\n    print(\"✅ Dataset ready: 150 train images (40 water background negatives), 40 val images (10 negatives)!\")\nelse:\n    print(f\"✅ Existing dataset detected: {len(train_imgs)} training images.\")

In [ ]:
# Step 6: Train YOLOv5n (Nano) on Drain Waste (50 Epochs)
# Pretrained yolov5n.pt provides rapid transfer learning convergence on GPU (~10-15 mins)
%cd /content/yolov5
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 50 \
  --data data/drain_waste.yaml \
  --weights yolov5n.pt \
  --cfg models/yolov5n.yaml \
  --project runs_drain_waste \
  --name exp_rv1103

In [ ]:
# Step 7: Export to ONNX with Rockchip RV1103 / RKNN NPU Subgraph Optimization
# Note: --rknpu is a flag (do NOT append 'rv1103' to --rknpu)
%cd /content/yolov5
!python export.py \
  --weights runs_drain_waste/exp_rv1103/weights/best.pt \
  --include onnx \
  --rknpu

print("\n[SUCCESS] Exported best.onnx optimized for Rockchip RV1103 NPU!")

In [ ]:
# Step 8: Download the trained ONNX model to your computer
import os
from google.colab import files

onnx_path = 'runs_drain_waste/exp_rv1103/weights/best.onnx'
if os.path.exists(onnx_path):
    print(f"Downloading {onnx_path} to your computer...")
    files.download(onnx_path)
else:
    print(f"[ERROR] '{onnx_path}' not found. Please check Step 6 and Step 7 output!")